In [27]:
from dataclasses import dataclass, field
from typing import Literal, Optional
import hashlib
from firecrawl import FirecrawlApp
from firecrawl.v2.types import ScrapeOptions
import re
import tiktoken
from abc import ABC, abstractmethod
from typing import List
from dataclasses import asdict
import json
from collections import Counter

SegmentType = Literal["text", "code", "table", "list"]
SourceType = Literal["epub", "website"]


In [28]:
@dataclass
class RawDocument:
    """Output of any ingestor."""
    doc_id: str            # chapter id or URL
    source_name: str       # tutorialspoint | litmentor | epub | web
    source_type: SourceType
    content: str           # raw HTML or Markdown
    title: str = ""
    metadata: dict = field(default_factory=dict)

@dataclass
class Segment:
    """Atomic parsed unit."""
    doc_id: str
    source_name: str
    title: str
    segment_type: SegmentType
    content: str
    heading_context: str = ""
    token_count: int = 0

@dataclass
class Chunk:
    """Final embeddable unit."""
    id: str           # chunk_00001
    source: str       # tutorialspoint | litmentor | epub | web
    url: str          # page URL or epub chapter id
    title: str        # page/chapter title
    heading: str      # section heading within page
    content: str      # clean markdown — the only text field (embedded + sent to LLM)
    has_code: bool
    tokens: int

    @property
    def content_hash(self) -> str:
        return hashlib.md5(self.content.encode()).hexdigest()

In [29]:
class BaseIngestor(ABC):
    @abstractmethod
    def ingest(self) -> List[RawDocument]:
        ...

In [30]:
class EpubIngestor(BaseIngestor):

    def __init__(self, path: str, source_name: str = "epub"):
        self.path = path
        self.source_name = source_name

    def ingest(self) -> List[RawDocument]:
        import ebooklib
        from ebooklib import epub
        book = epub.read_epub(self.path)
        docs = []
        for item in book.get_items():
            if item.get_type() != ebooklib.ITEM_DOCUMENT:
                continue
            chapter_id = item.get_id()
            if chapter_id.lower().startswith("chapter-"):
                docs.append(RawDocument(
                    doc_id=chapter_id,
                    source_name=self.source_name,
                    source_type="epub",
                    content=item.get_body_content().decode("utf-8", errors="replace"),
                    title=item.get_name(),
                    metadata={"book_title": book.title},
                ))
        return docs

In [31]:
class WebsiteIngestor(BaseIngestor):
    """
    Crawls an entire website using Firecrawl.
    Returns one RawDocument per page, content is Markdown.
    """
    def __init__(self, start_url: str,
                 source_name: str = "web",
                 api_url: str = "http://localhost:3002",
                 limit: int = 200,
                 exclude_patterns: Optional[List[str]] = None):
        self.start_url = start_url
        self.source_name = source_name
        self.api_url = api_url
        self.limit = limit
        self.exclude_patterns = exclude_patterns or []

    def ingest(self) -> List[RawDocument]:
        fc = FirecrawlApp(api_url=self.api_url)

        crawl_result = fc.crawl(
            self.start_url,
            limit=self.limit,
            scrape_options=ScrapeOptions(
                formats=["markdown"],
                only_main_content=True,
                exclude_tags=["nav", "footer", "aside", ".sidebar",
                               ".advertisement", ".google-auto-placed",
                               ".site-header", ".nav-links"],
            ),
            exclude_paths=self.exclude_patterns,
        )

        docs = []
        for page in crawl_result.data:
            url = page.metadata.source_url
            markdown = page.markdown or ""
            if not markdown.strip():
                continue
            docs.append(RawDocument(
                doc_id=url,
                source_name=self.source_name,
                source_type="website",
                content=markdown,
                title=page.metadata.title or "",
                metadata={
                    "description": page.metadata.description or "",
                    "status_code": page.metadata.status_code,
                }
            ))
        return docs


In [32]:
class TutorialspointIngestor(BaseIngestor):
    def __init__(self, urls: List[str], source_name: str = "tutorialspoint"):
        self.urls = urls
        self.source_name = source_name

    def ingest(self) -> List[RawDocument]:
        from firecrawl import FirecrawlApp

        fc = FirecrawlApp(api_url="http://localhost:3002")

        batch_result = fc.batch_scrape(
            self.urls,
            only_main_content=True,
            exclude_tags=["nav", "footer", ".sidebar",
                          ".advertisement", ".google-auto-placed"],
        )

        docs = []
        for page in batch_result.data:
            url = page.metadata.source_url or page.metadata.url or ""
            markdown = page.markdown or ""
            if not markdown.strip():
                continue
            docs.append(RawDocument(
                doc_id=url,
                source_name=self.source_name,
                source_type="website",
                content=markdown,
                title=page.metadata.title or "",
            ))

        print(f"  → {len(docs)}/{len(self.urls)} páginas scrapeadas")
        return docs


In [33]:
class LitmentorIngestor(BaseIngestor):
    def __init__(self, urls: List[str], source_name: str = "litmentor"):
        self.urls = urls
        self.source_name = source_name

    def ingest(self) -> List[RawDocument]:
        from firecrawl import FirecrawlApp

        fc = FirecrawlApp(api_url="http://localhost:3002")

        batch_result = fc.batch_scrape(
            self.urls,
            only_main_content=True,
            exclude_tags=["nav", "footer", ".col-3", ".advertisement", ".google-auto-placed"],
        )

        docs = []
        for page in batch_result.data:
            url = page.metadata.source_url or page.metadata.url or ""
            markdown = page.markdown or ""
            if not markdown.strip():
                continue
            markdown = markdown.replace('\\\\', '\\')
            docs.append(RawDocument(
                doc_id=url,
                source_name=self.source_name,
                source_type="website",
                content=markdown,
                title=page.metadata.title or "",
            ))

        print(f"  → {len(docs)}/{len(self.urls)} páginas scrapeadas")
        return docs


In [34]:
TOKENIZER = tiktoken.get_encoding("cl100k_base")

MAX_CHUNK_TOKENS = 2048

_NOMIC_HARD_LIMIT = 8192
_EMBED_PREFIX = "search_document: "
_EMBED_PREFIX_TOKENS = len(TOKENIZER.encode(_EMBED_PREFIX))

EMBED_MAX_TOKENS = _NOMIC_HARD_LIMIT - _EMBED_PREFIX_TOKENS - 50  # = 8139, safe margin

def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))

def count_embed_tokens(text: str) -> int: 
    """Count tokens as nomic-embed-text will see them (prefix included)."""
    return _EMBED_PREFIX_TOKENS + count_tokens(text)

def exceeds_embed_limit(text: str) -> bool:
    """Returns True if the text would exceed the safe embed limit (prefix included)."""
    return count_embed_tokens(text) > EMBED_MAX_TOKENS

_SETEXT_RE = re.compile(r'^[=\-]{2,}\s*$')

class BaseParser(ABC):
    @abstractmethod
    def parse(self, doc: RawDocument) -> List[Segment]:
        ...

class MarkdownParser(BaseParser):
    """
    Parses Firecrawl markdown output into Segments.
    Handles: ATX headings (#), Setext headings (===, ---),
             fenced code blocks (``` and ~~~),
             4-space/tab indented code blocks, and paragraphs.
    """

    def parse(self, doc: RawDocument) -> List[Segment]:
        segments = []
        current_heading = ""
        # Strip markdown image syntax (e.g. primis base64 blobs) - no semantic value
        content = re.sub(r'!\[([^\]]*)\]\([^)]*\)', r'\1', doc.content)
        lines = content.splitlines()
        i = 0

        while i < len(lines):
            line = lines[i]

            heading_match = re.match(r'^(#{1,4})\s+(.*)', line)
            if heading_match:
                current_heading = heading_match.group(2).strip()
                i += 1
                continue

            if (i + 1 < len(lines)
                    and line.strip()
                    and not line.startswith("    ") and not line.startswith("\t")
                    and _SETEXT_RE.match(lines[i + 1].strip())):
                current_heading = line.strip()
                i += 2
                continue

            fence_match = re.match(r'^(`{3,}|~{3,})', line.strip())
            if fence_match:
                fence_char = fence_match.group(1)[0]
                code_lines = []
                i += 1
                while i < len(lines) and not lines[i].strip().startswith(fence_char * 3):
                    code_lines.append(lines[i])
                    i += 1
                i += 1  # skip closing fence
                code = "\n".join(code_lines).strip()
                if code:
                    segments.append(Segment(
                        doc_id=doc.doc_id,
                        source_name=doc.source_name,
                        title=doc.title,
                        segment_type="code",
                        content=code,
                        heading_context=current_heading,
                        token_count=count_tokens(code),
                    ))
                continue

            if line.startswith("    ") or line.startswith("\t"):
                code_lines = []
                while i < len(lines) and (lines[i].startswith("    ") or lines[i].startswith("\t") or lines[i].strip() == ""):
                    stripped = lines[i][4:] if lines[i].startswith("    ") else lines[i].lstrip("\t")
                    code_lines.append(stripped if lines[i].strip() else "")
                    i += 1
                code = "\n".join(code_lines).strip()
                if code:
                    segments.append(Segment(
                        doc_id=doc.doc_id,
                        source_name=doc.source_name,
                        title=doc.title,
                        segment_type="code",
                        content=code,
                        heading_context=current_heading,
                        token_count=count_tokens(code),
                    ))
                continue

            if line.strip() and not _SETEXT_RE.match(line.strip()):
                para_lines = [line.strip()]
                i += 1
                while (i < len(lines) and lines[i].strip()
                       and not lines[i].startswith("#")
                       and not re.match(r'^(`{3,}|~{3,})', lines[i].strip())
                       and not lines[i].startswith("    ")
                       and not lines[i].startswith("\t")
                       and not _SETEXT_RE.match(lines[i].strip())
                       and not (i + 1 < len(lines) and _SETEXT_RE.match(lines[i + 1].strip()))):
                    para_lines.append(lines[i].strip())
                    i += 1

                text = " ".join(para_lines)
                sub_texts = []
                current, current_tokens = [], 0
                for sentence in re.split(r'(?<=[.!?])\s+', text):
                    t = count_tokens(sentence)
                    if current_tokens + t > MAX_CHUNK_TOKENS and current:
                        sub_texts.append(" ".join(current))
                        current, current_tokens = [sentence], t
                    else:
                        current.append(sentence)
                        current_tokens += t
                if current:
                    sub_texts.append(" ".join(current))

                for sub_text in sub_texts:
                    if 30 < len(sub_text) <= 20000:
                        segments.append(Segment(
                            doc_id=doc.doc_id,
                            source_name=doc.source_name,
                            title=doc.title,
                            segment_type="text",
                            content=sub_text,
                            heading_context=current_heading,
                            token_count=count_tokens(sub_text),
                        ))
                continue

            i += 1

        return segments


class HtmlParser(BaseParser):
    """
    Parses EPUB HTML into Segments.
    """

    def parse(self, doc: RawDocument) -> List[Segment]:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(doc.content, "html.parser")
        segments = []
        current_heading = ""
        body = soup.body or soup

        for element in body.find_all(["h1","h2","h3","h4","pre","p"], recursive=True):
            if element.name in ("h1","h2","h3","h4"):
                current_heading = element.get_text(strip=True)

            elif element.name == "pre":
                code = element.get_text().strip()
                if code:
                    segments.append(Segment(
                        doc_id=doc.doc_id,
                        source_name=doc.source_name,
                        title=doc.title,
                        segment_type="code",
                        content=code,
                        heading_context=current_heading,
                        token_count=count_tokens(code),
                    ))
            elif element.name == "p":
                if element.find_parent("pre"):
                    continue
                text = element.get_text(separator=" ", strip=True)
                if len(text) > 30:
                    sub_texts = []
                    current, current_tokens = [], 0
                    for sentence in re.split(r'(?<=[.!?])\s+', text):
                        t = count_tokens(sentence)
                        if current_tokens + t > MAX_CHUNK_TOKENS and current:
                            sub_texts.append(" ".join(current))
                            current, current_tokens = [sentence], t
                        else:
                            current.append(sentence)
                            current_tokens += t
                    if current:
                        sub_texts.append(" ".join(current))

                    for sub_text in sub_texts:
                        if 30 < len(sub_text) <= 20000:
                            segments.append(Segment(
                                doc_id=doc.doc_id,
                                source_name=doc.source_name,
                                title=doc.title,
                                segment_type="text",
                                content=sub_text,
                                heading_context=current_heading,
                                token_count=count_tokens(sub_text),
                            ))

        return segments


In [35]:
class SemanticChunker:
    """
    Groups Segments into token-budget-aware Chunks.
    Strategy:
      - A code segment anchors a chunk; surrounding text is added until budget fills.
      - Pure text segments are grouped greedily up to budget.
    Content is emitted as clean markdown (## heading + fenced ```c blocks).
    Chunks are guaranteed to never exceed nomic's hard limit (prefix included)
    via post-build enforcement.
    """

    def __init__(self, max_tokens: int = MAX_CHUNK_TOKENS, source_name: str = "epub"):
        self.max_tokens = min(max_tokens, EMBED_MAX_TOKENS)
        self.source_name = source_name
        self._counter = 0

    def build_chunks(self, segments: List[Segment]) -> List[Chunk]:
        chunks = []
        i = 0

        while i < len(segments):
            seg = segments[i]
            if seg.segment_type == "code":
                if exceeds_embed_limit(seg.content):
                    chunks.extend(self._split_oversized_segment(seg))
                else:
                    chunks.append(self._build_code_chunk(segments, i))
                i += 1
            else:
                if exceeds_embed_limit(seg.content):
                    chunks.extend(self._split_oversized_segment(seg))
                    i += 1
                else:
                    chunk, consumed = self._build_text_chunk(segments, i)
                    chunks.append(chunk)
                    i += consumed

        enforced = []
        for chunk in chunks:
            if not exceeds_embed_limit(chunk.content):
                enforced.append(chunk)
            else:
                enforced.extend(self._enforce_limit(chunk))

        return self._deduplicate(enforced)

    def _build_code_chunk(self, segments: List[Segment], code_idx: int) -> Chunk:
        seg = segments[code_idx]
        _fence_overhead = count_tokens("```c\n\n```")  # ~5 tokens for the fenced block markers
        _heading_overhead = count_tokens(f"## {seg.heading_context}\n\n") if seg.heading_context else 0
        budget = self.max_tokens - seg.token_count - _fence_overhead - _heading_overhead
        budget = max(0, budget)
        before_texts, after_texts = [], []

        j = code_idx - 1
        while j >= 0 and segments[j].segment_type == "text" and budget > 0:
            if segments[j].token_count <= budget:
                before_texts.insert(0, segments[j].content)
                budget -= segments[j].token_count
            j -= 1
            break

        j = code_idx + 1
        while j < len(segments) and segments[j].segment_type == "text" and budget > 0:
            if segments[j].token_count <= budget:
                after_texts.append(segments[j].content)
                budget -= segments[j].token_count
            j += 1
            break

        surrounding = " ".join(before_texts + after_texts)
        parts = []
        if seg.heading_context:
            parts.append(f"## {seg.heading_context}")
        if surrounding:
            parts.append(surrounding)
        parts.append(f"```c\n{seg.content}\n```")
        content = "\n\n".join(parts)
        return self._make_chunk(seg, content, True)

    def _build_text_chunk(self, segments: List[Segment], start: int):
        group = []
        i = start

        while i < len(segments) and segments[i].segment_type == "text":
            s = segments[i]
            candidate_group = group + [s.content]
            parts = []
            if segments[start].heading_context:
                parts.append(f"## {segments[start].heading_context}")
            parts.append(" ".join(candidate_group))
            candidate_content = "\n\n".join(parts)

            if exceeds_embed_limit(candidate_content):
                break

            group.append(s.content)
            i += 1

        if not group:
            group.append(segments[start].content)
            i = start + 1

        text = " ".join(group)
        parts = []
        if segments[start].heading_context:
            parts.append(f"## {segments[start].heading_context}")
        parts.append(text)
        content = "\n\n".join(parts)
        consumed = max(1, i - start)
        return self._make_chunk(segments[start], content, False), consumed

    def _split_oversized_segment(self, seg: Segment) -> List["Chunk"]:
        """
        Last-resort split for a single segment that alone exceeds the nomic limit.
        Strategy: try sentence boundaries first, then newline boundaries, then
        word-by-word token-trim as a last resort.
        Budget is calculated as: EMBED_MAX_TOKENS - prefix_tokens - heading_tokens.
        """
        heading_tokens = count_tokens(f"## {seg.heading_context}\n\n") if seg.heading_context else 0
        budget = EMBED_MAX_TOKENS - _EMBED_PREFIX_TOKENS - heading_tokens

        candidates = re.split(r'(?<=[.!?])\s+', seg.content)
        used_newlines = False
        if len(candidates) == 1:
            candidates = seg.content.splitlines(keepends=True)
            used_newlines = True
        if len(candidates) == 1 or all(count_tokens(c) > budget for c in candidates):
            candidates = seg.content.split()
            used_newlines = False

        sub_chunks: List[Chunk] = []
        current_parts: List[str] = []
        current_tokens = 0

        for part in candidates:
            t = count_tokens(part)
            if t > budget:
                encoded = TOKENIZER.encode(part)
                part = TOKENIZER.decode(encoded[:budget])
                t = budget
            if current_tokens + t > budget and current_parts:
                sep = "" if used_newlines else " "
                content = self._format_content(seg, sep.join(current_parts), seg.segment_type == "code")
                sub_chunks.append(self._make_chunk(seg, content, seg.segment_type == "code"))
                current_parts = [part]
                current_tokens = t
            else:
                current_parts.append(part)
                current_tokens += t

        if current_parts:
            sep = "" if used_newlines else " "
            content = self._format_content(seg, sep.join(current_parts), seg.segment_type == "code")
            sub_chunks.append(self._make_chunk(seg, content, seg.segment_type == "code"))

        return sub_chunks

    def _enforce_limit(self, chunk: Chunk) -> List[Chunk]:
        """
        Post-build safety net: split a fully-formed Chunk that still exceeds
        EMBED_MAX_TOKENS (prefix included). Separates heading prefix, splits
        body on sentence then newline then word boundaries, re-attaches heading
        to every sub-chunk.
        """
        heading_prefix = ""
        body = chunk.content
        if chunk.content.startswith("## "):
            parts = chunk.content.split("\n\n", 1)
            if len(parts) == 2:
                heading_prefix = parts[0] + "\n\n"
                body = parts[1]

        heading_tokens = count_tokens(heading_prefix)
        budget = EMBED_MAX_TOKENS - _EMBED_PREFIX_TOKENS - heading_tokens

        candidates = re.split(r'(?<=[.!?])\s+', body)
        used_newlines = False
        if len(candidates) == 1:
            candidates = body.splitlines(keepends=True)
            used_newlines = True
        if len(candidates) == 1 or all(count_tokens(c) > budget for c in candidates):
            candidates = body.split()
            used_newlines = False

        sub_chunks: List[Chunk] = []
        current_parts: List[str] = []
        current_tokens = 0

        for part in candidates:
            t = count_tokens(part)
            if t > budget:
                encoded = TOKENIZER.encode(part)
                part = TOKENIZER.decode(encoded[:budget])
                t = budget
            if current_tokens + t > budget and current_parts:
                sep = "" if used_newlines else " "
                content = heading_prefix + sep.join(current_parts)
                sub_chunks.append(self._make_chunk_from_chunk(chunk, content))
                current_parts = [part]
                current_tokens = t
            else:
                current_parts.append(part)
                current_tokens += t

        if current_parts:
            sep = "" if used_newlines else " "
            content = heading_prefix + sep.join(current_parts)
            sub_chunks.append(self._make_chunk_from_chunk(chunk, content))

        return sub_chunks

    def _make_chunk_from_chunk(self, ref: Chunk, content: str) -> Chunk:
        """Create a new Chunk reusing metadata from an existing Chunk."""
        cid = f"chunk_{self._counter:05d}"
        self._counter += 1
        return Chunk(
            id=cid,
            source=ref.source,
            url=ref.url,
            title=ref.title,
            heading=ref.heading,
            content=content,
            has_code=ref.has_code,
            tokens=count_tokens(content),
        )

    def _format_content(self, seg: Segment, text: str, is_code: bool) -> str:
        parts = []
        if seg.heading_context:
            parts.append(f"## {seg.heading_context}")
        parts.append(f"```c\n{text}\n```" if is_code else text)
        return "\n\n".join(parts)

    def _make_chunk(self, ref_seg: Segment, content: str, has_code: bool) -> Chunk:
        cid = f"chunk_{self._counter:05d}"
        self._counter += 1
        return Chunk(
            id=cid,
            source=ref_seg.source_name,
            url=ref_seg.doc_id,
            title=ref_seg.title,
            heading=ref_seg.heading_context,
            content=content,
            has_code=has_code,
            tokens=count_tokens(content),
        )

    @staticmethod
    def _deduplicate(chunks: List[Chunk]) -> List[Chunk]:
        seen, unique = set(), []
        for c in chunks:
            h = c.content_hash
            if h not in seen:
                seen.add(h)
                unique.append(c)
        return unique

In [36]:

def run_all_sources(
    sources: list,
    output_path: str,
    max_chunk_tokens: int = 2048,
    min_chunk_tokens: int = 80,
) -> List[Chunk]:
    """
    Runs the full pipeline for every source and merges into a single JSON.
    Each source is a dict: {ingestor, parser_cls}
    """
    all_chunks: List[Chunk] = []
    global_counter = 0

    for cfg in sources:
        ingestor  = cfg["ingestor"]
        parser_cls = cfg["parser_cls"]
        print(f"\n── {type(ingestor).__name__} ──")

        print("  Step 1: Ingesting...")
        raw_docs = ingestor.ingest()
        print(f"    → {len(raw_docs)} documents")

        print("  Step 2: Parsing...")
        parser = parser_cls()
        all_segments = []
        for doc in raw_docs:
            all_segments.extend(parser.parse(doc))
        print(f"    → {len(all_segments)} segments")

        # Noise filter
        all_segments = [s for s in all_segments
                        if not re.search(r"(www\.|ISBN|©|All rights reserved)", s.content, re.I)]

        print("  Step 3: Chunking...")
        source_name = raw_docs[0].source_name if raw_docs else "unknown"
        chunker = SemanticChunker(max_tokens=max_chunk_tokens, source_name=source_name)
        chunker._counter = global_counter
        chunks = chunker.build_chunks(all_segments)
        global_counter = chunker._counter

        # Drop micro-chunks (text-only below min threshold)
        before = len(chunks)
        chunks = [c for c in chunks if c.has_code or c.tokens >= min_chunk_tokens]
        print(f"    → {len(chunks)} chunks  (dropped {before - len(chunks)} micro-chunks)")
        print(f"       With code : {sum(1 for c in chunks if c.has_code)}")
        print(f"       Text only : {sum(1 for c in chunks if not c.has_code)}")

        # ── Verify no chunk exceeds embed limit (prefix included) ────────────
        violations = [c for c in chunks if exceeds_embed_limit(c.content)]
        if violations:
            print(f"    ⛔ {len(violations)} chunks exceed nomic hard limit (prefix included)!")
            for v in violations:
                print(f"       {v.id} | content={v.tokens}t | embed={count_embed_tokens(v.content)}t")
        else:
            print("    ✔️  All chunks within nomic embed limit (prefix included)")

        all_chunks.extend(chunks)

    # Global dedup across sources
    seen, unique = set(), []
    for c in all_chunks:
        h = c.content_hash
        if h not in seen:
            seen.add(h)
            unique.append(c)

    print("\n── TOTAL ──")
    print(f"  {len(unique)} chunks across {len(sources)} sources")
    dist = Counter(c.source for c in unique)
    for src, n in dist.items():
        print(f"    {src}: {n}")

    oversized = [c for c in unique if count_embed_tokens(c.content) > EMBED_MAX_TOKENS]
    if oversized:
        print(f"  ⚠️  {len(oversized)} chunks exceed EMBED_MAX_TOKENS ({EMBED_MAX_TOKENS}t) — review parser")
        for v in oversized:
            print(f"     {v.id} | embed={count_embed_tokens(v.content)}t | chars={len(v.content)}")

    print("\n  Exporting...")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump([asdict(c) for c in unique], f, ensure_ascii=False, indent=2)
    print(f"  → Saved to {output_path}")

    return unique


In [37]:
TUTORIALSPOINT_C_URLS = [
    # Basics
    "https://www.tutorialspoint.com/cprogramming/c_overview.htm",
    "https://www.tutorialspoint.com/cprogramming/c_features.htm",
    "https://www.tutorialspoint.com/cprogramming/c_history.htm",
    "https://www.tutorialspoint.com/cprogramming/c_standards.htm",
    "https://www.tutorialspoint.com/cprogramming/c_environment_setup.htm",
    "https://www.tutorialspoint.com/cprogramming/c_program_structure.htm",
    "https://www.tutorialspoint.com/cprogramming/c_hello_world.htm",
    "https://www.tutorialspoint.com/cprogramming/c_compilation_process.htm",
    "https://www.tutorialspoint.com/cprogramming/c_comments.htm",
    "https://www.tutorialspoint.com/cprogramming/c_basic_syntax.htm",
    "https://www.tutorialspoint.com/cprogramming/c_user_input.htm",
    "https://www.tutorialspoint.com/cprogramming/c_printf_function.htm",
    "https://www.tutorialspoint.com/cprogramming/c_format_specifiers.htm",
    # Lexical Elements
    "https://www.tutorialspoint.com/cprogramming/c_tokens.htm",
    "https://www.tutorialspoint.com/cprogramming/c_keywords.htm",
    "https://www.tutorialspoint.com/cprogramming/c_identifiers.htm",
    # Variables and Constants
    "https://www.tutorialspoint.com/cprogramming/c_variables.htm",
    "https://www.tutorialspoint.com/cprogramming/c_constants.htm",
    "https://www.tutorialspoint.com/cprogramming/c_const_qualifier.htm",
    "https://www.tutorialspoint.com/cprogramming/c_internal_and_external_linkage.htm",
    # Data Types
    "https://www.tutorialspoint.com/cprogramming/c_data_types.htm",
    "https://www.tutorialspoint.com/cprogramming/c_literals.htm",
    "https://www.tutorialspoint.com/cprogramming/c_escape_sequences.htm",
    "https://www.tutorialspoint.com/cprogramming/c_booleans.htm",
    "https://www.tutorialspoint.com/cprogramming/c_integer_promotions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_character_arithmetic.htm",
    "https://www.tutorialspoint.com/cprogramming/c_type_conversion.htm",
    "https://www.tutorialspoint.com/cprogramming/c_type_casting.htm",
    # Operators
    "https://www.tutorialspoint.com/cprogramming/c_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_arithmetic_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_unary_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_relational_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_logical_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_bitwise_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_assignment_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_increment_and_decrement_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_ternary_operator.htm",
    "https://www.tutorialspoint.com/cprogramming/c_sizeof_operator.htm",
    "https://www.tutorialspoint.com/cprogramming/c_operators_precedence.htm",
    "https://www.tutorialspoint.com/cprogramming/c_misc_operators.htm",
    # Decision Making
    "https://www.tutorialspoint.com/cprogramming/c_decision_making.htm",
    "https://www.tutorialspoint.com/cprogramming/if_statement_in_c.htm",
    "https://www.tutorialspoint.com/cprogramming/if_else_statement_in_c.htm",
    "https://www.tutorialspoint.com/cprogramming/c_if_else_if_ladder.htm",
    "https://www.tutorialspoint.com/cprogramming/nested_if_statements_in_c.htm",
    "https://www.tutorialspoint.com/cprogramming/switch_statement_in_c.htm",
    "https://www.tutorialspoint.com/cprogramming/nested_switch_statements_in_c.htm",
    "https://www.tutorialspoint.com/cprogramming/c_switch_case_using_range.htm",
    # Loops
    "https://www.tutorialspoint.com/cprogramming/c_loops.htm",
    "https://www.tutorialspoint.com/cprogramming/c_for_loop.htm",
    "https://www.tutorialspoint.com/cprogramming/c_while_loop.htm",
    "https://www.tutorialspoint.com/cprogramming/c_do_while_loop.htm",
    "https://www.tutorialspoint.com/cprogramming/c_for_loop_vs_while_loop.htm",
    "https://www.tutorialspoint.com/cprogramming/c_nested_loops.htm",
    "https://www.tutorialspoint.com/cprogramming/c_infinite_loop.htm",
    "https://www.tutorialspoint.com/cprogramming/c_break_statement.htm",
    "https://www.tutorialspoint.com/cprogramming/c_continue_statement.htm",
    "https://www.tutorialspoint.com/cprogramming/c_goto_statement.htm",
    # Functions
    "https://www.tutorialspoint.com/cprogramming/c_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_functions_prototype.htm",
    "https://www.tutorialspoint.com/cprogramming/c_main_function.htm",
    "https://www.tutorialspoint.com/cprogramming/c_function_call_by_value.htm",
    "https://www.tutorialspoint.com/cprogramming/c_function_call_by_reference.htm",
    "https://www.tutorialspoint.com/cprogramming/c_nested_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_variadic_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_user_defined_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_callback_function.htm",
    "https://www.tutorialspoint.com/cprogramming/c_return_statement.htm",
    "https://www.tutorialspoint.com/cprogramming/c_recursion.htm",
    "https://www.tutorialspoint.com/cprogramming/c_predefined_identifier_func.htm",
    # Scope
    "https://www.tutorialspoint.com/cprogramming/c_scope_rules.htm",
    "https://www.tutorialspoint.com/cprogramming/c_static_variables.htm",
    "https://www.tutorialspoint.com/cprogramming/c_global_variables.htm",
    # Arrays
    "https://www.tutorialspoint.com/cprogramming/c_arrays.htm",
    "https://www.tutorialspoint.com/cprogramming/c_properties_of_array.htm",
    "https://www.tutorialspoint.com/cprogramming/c_multi_dimensional_arrays.htm",
    "https://www.tutorialspoint.com/cprogramming/c_passing_arrays_to_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_return_arrays_from_function.htm",
    "https://www.tutorialspoint.com/cprogramming/c_variable_length_arrays.htm",
    "https://www.tutorialspoint.com/cprogramming/c_dynamic_arrays.htm",
    # Strings
    "https://www.tutorialspoint.com/cprogramming/c_strings.htm",
    "https://www.tutorialspoint.com/cprogramming/c_array_of_strings.htm",
    "https://www.tutorialspoint.com/cprogramming/c_character_arrays.htm",
    "https://www.tutorialspoint.com/cprogramming/c_special_characters.htm",
    # Pointers
    "https://www.tutorialspoint.com/cprogramming/c_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_initialization_of_pointer_arrays.htm",
    "https://www.tutorialspoint.com/cprogramming/c_applications_of_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_dereference_pointer.htm",
    "https://www.tutorialspoint.com/cprogramming/c_null_pointer.htm",
    "https://www.tutorialspoint.com/cprogramming/c_void_pointer.htm",
    "https://www.tutorialspoint.com/cprogramming/c_constant_pointers_and_pointer_to_constant.htm",
    "https://www.tutorialspoint.com/cprogramming/c_dangling_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pointer_arithmetic.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pointers_and_arrays.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pointer_vs_array.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pointer_to_an_array.htm",
    "https://www.tutorialspoint.com/cprogramming/c_array_of_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pointers_vs_multi_dimensional_arrays.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pointer_to_pointer.htm",
    "https://www.tutorialspoint.com/cprogramming/c_chain_of_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_character_pointers_and_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_passing_pointers_to_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_return_pointer_from_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_function_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_array_of_function_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pointers_to_structures.htm",
    "https://www.tutorialspoint.com/cprogramming/c_near_far_and_huge_pointers.htm",
    "https://www.tutorialspoint.com/cprogramming/c_restrict_keyword.htm",
    # User-Defined Data Types
    "https://www.tutorialspoint.com/cprogramming/c_structures.htm",
    "https://www.tutorialspoint.com/cprogramming/c_structures_and_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_arrays_of_structures.htm",
    "https://www.tutorialspoint.com/cprogramming/c_self_referential_structures.htm",
    "https://www.tutorialspoint.com/cprogramming/c_dot_operator.htm",
    "https://www.tutorialspoint.com/cprogramming/c_lookup_tables.htm",
    "https://www.tutorialspoint.com/cprogramming/c_enumeration_or_enum.htm",
    "https://www.tutorialspoint.com/cprogramming/c_structure_padding_and_packing.htm",
    "https://www.tutorialspoint.com/cprogramming/c_nested_structures.htm",
    "https://www.tutorialspoint.com/cprogramming/c_anonymous_structures_and_unions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_unions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_bit_fields.htm",
    "https://www.tutorialspoint.com/cprogramming/c_typedef.htm",
    "https://www.tutorialspoint.com/cprogramming/c_flexible_array_members_in_structures.htm",
    "https://www.tutorialspoint.com/cprogramming/c_structures_vs_unions.htm",
    # Memory Management
    "https://www.tutorialspoint.com/cprogramming/c_memory_layout.htm",
    "https://www.tutorialspoint.com/cprogramming/c_memory_management.htm",
    "https://www.tutorialspoint.com/cprogramming/c_memory_address.htm",
    "https://www.tutorialspoint.com/cprogramming/c_storage_classes.htm",
    "https://www.tutorialspoint.com/cprogramming/c_dynamic_array_resizing.htm",
    "https://www.tutorialspoint.com/cprogramming/c_memory_leaks.htm",
    # File Handling
    "https://www.tutorialspoint.com/cprogramming/c_file_io.htm",
    "https://www.tutorialspoint.com/cprogramming/c_input_output.htm",
    "https://www.tutorialspoint.com/cprogramming/c_file_operations.htm",
    "https://www.tutorialspoint.com/cprogramming/c_formatted_output.htm",
    "https://www.tutorialspoint.com/cprogramming/c_getc_getchar_getch_getche.htm",
    # Preprocessors
    "https://www.tutorialspoint.com/cprogramming/c_preprocessors.htm",
    "https://www.tutorialspoint.com/cprogramming/c_pragmas.htm",
    "https://www.tutorialspoint.com/cprogramming/c_macros.htm",
    "https://www.tutorialspoint.com/cprogramming/c_working_of_preprocessor.htm",
    "https://www.tutorialspoint.com/cprogramming/c_preprocessor_operators.htm",
    "https://www.tutorialspoint.com/cprogramming/c_header_files.htm",
    "https://www.tutorialspoint.com/cprogramming/c_custom_header_files.htm",
    # Miscellaneous
    "https://www.tutorialspoint.com/cprogramming/c_error_handling.htm",
    "https://www.tutorialspoint.com/cprogramming/c_variable_arguments.htm",
    "https://www.tutorialspoint.com/cprogramming/c_command_execution.htm",
    "https://www.tutorialspoint.com/cprogramming/c_math_functions.htm",
    "https://www.tutorialspoint.com/cprogramming/c_static_keyword.htm",
    "https://www.tutorialspoint.com/cprogramming/c_random_number_generation.htm",
    "https://www.tutorialspoint.com/cprogramming/c_command_line_arguments.htm",
]

In [38]:
LITMENTOR_C_URLS = [
    # Basic
    "http://www.litmentor.com/learn-c-tutorial/introduction-of-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/advantages-and-disadvantages-of-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/history-of-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/first-c-program.php",
    "http://www.litmentor.com/learn-c-tutorial/c-format-specifiers.php",
    "http://www.litmentor.com/learn-c-tutorial/comment-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/c-character-set.php",
    "http://www.litmentor.com/learn-c-tutorial/constant-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/variable-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/key-words-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/identifiers-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/data-type-in-c-programming-language.php",
    # Operators
    "http://www.litmentor.com/learn-c-tutorial/operators-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/arithmetic-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/relational-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/logical-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/increment-and-decrement-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/conditional-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/bitwise-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/assignment-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/sizeof-operators-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/comma-operator-c-language.php",
    # Casting Operators skipped — placeholder URL (#)
    # Control Structure
    "http://www.litmentor.com/learn-c-tutorial/control-structures-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/if-statement-in-c-programming.php",
    "http://www.litmentor.com/learn-c-tutorial/if-else-statement-in-c-programming.php",
    "http://www.litmentor.com/learn-c-tutorial/else-if-statement-in-c-programming.php",
    "http://www.litmentor.com/learn-c-tutorial/nested-else-statement-in-c-programming.php",
    "http://www.litmentor.com/learn-c-tutorial/switch-statement-c-programming-language.php",
    # Loop
    "http://www.litmentor.com/learn-c-tutorial/loops-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/for-loop-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/while-loop-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/do-while-loop-in-c-programming-language.php",
    # Branch & Jump
    "http://www.litmentor.com/learn-c-tutorial/break-statement-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/continue-statement-in-c-language.php",
    "http://www.litmentor.com/learn-c-tutorial/goto-statement-in-c-programming.php",
    # Array (only pages with real URLs; remaining sub-topics are placeholder #)
    "http://www.litmentor.com/learn-c-tutorial/array-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/types-of-array-in-c.php",
    "http://www.litmentor.com/learn-c-tutorial/array-declaration-in-c.php",
    "http://www.litmentor.com/learn-c-tutorial/array-Initialization-in-c.php",
    # Multi-dimensional Array sub-topics all placeholder (#) — skipped
    # Pointers, Functions, Strings, Structures, etc.
    "http://www.litmentor.com/learn-c-tutorial/Pointers-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/functions-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/string-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/string-practice-questions-in-c/string-practice.php",
    "http://www.litmentor.com/learn-c-tutorial/structure-and-union-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/union-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/enum-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/storage-classes-in-c-programming-language.php",
    "http://www.litmentor.com/learn-c-tutorial/input-output-in-c-programming-language.php",
]

In [39]:
all_chunks = run_all_sources(
    sources=[
        dict(
            ingestor=EpubIngestor("../data/rag/to_chuncking/mordern_c.epub"),
            parser_cls=HtmlParser,
        ),
        dict(
            ingestor=TutorialspointIngestor(urls=TUTORIALSPOINT_C_URLS),
            parser_cls=MarkdownParser,
        ),
        dict(
            ingestor=LitmentorIngestor(urls=LITMENTOR_C_URLS),
            parser_cls=MarkdownParser,
        ),
        dict(
            ingestor=WebsiteIngestor(
                start_url="https://www.how2lab.com/programming/c/",
                source_name="web",
                exclude_patterns=["/tag/*", "/category/*", "/author/*",
                                   "/search*", "/login*", "/signup*", "/about*"],
            ),
            parser_cls=MarkdownParser,
        ),
    ],
    output_path="../data/rag/to_ingest/chunks_all.json",
    max_chunk_tokens=2048,
    min_chunk_tokens=80,
)



── EpubIngestor ──
  Step 1: Ingesting...
    → 22 documents
  Step 2: Parsing...
    → 3258 segments
  Step 3: Chunking...
    → 922 chunks  (dropped 148 micro-chunks)
       With code : 548
       Text only : 374
    ✔️  All chunks within nomic embed limit (prefix included)

── TutorialspointIngestor ──
  Step 1: Ingesting...
  → 148/148 páginas scrapeadas
    → 148 documents
  Step 2: Parsing...
    → 31695 segments
  Step 3: Chunking...
    → 880 chunks  (dropped 130 micro-chunks)
       With code : 570
       Text only : 310
    ✔️  All chunks within nomic embed limit (prefix included)

── LitmentorIngestor ──
  Step 1: Ingesting...
  → 48/48 páginas scrapeadas
    → 48 documents
  Step 2: Parsing...
    → 1366 segments
  Step 3: Chunking...
    → 364 chunks  (dropped 116 micro-chunks)
       With code : 283
       Text only : 81
    ✔️  All chunks within nomic embed limit (prefix included)

── WebsiteIngestor ──
  Step 1: Ingesting...
    → 55 documents
  Step 2: Parsing...
    